In [1]:
import re
import pandas as pd
import os
import glob
import gzip
import matplotlib.pyplot as plt
import seaborn as sns
import json
import folium

# Read output file

In [2]:

log_path = r"C:\Users\Minamsj\FOMOsim\output_TD_0612251922.txt"

# Adjust this pattern to match how weights are written in your file.
# Examples it would match:
#   "weights = 0.3, 0.5, 0.2"
#   "Weights: [0.3, 0.5, 0.2]"
#weights_pattern = re.compile(r"weights?\s*[:=]\s*(.*)", re.IGNORECASE)
alpha_pattern = re.compile(r"([0-9]*\.[0-9]+)\s*$")
# station ID like S60, S103, etc.
station_pattern = re.compile(r"\bS(\d+)\b")
lost_trip_type_pattern = re.compile(r"\(([^,]+),\s*t=")

runs = []
current_run = None

with open(log_path, "r", encoding="utf-16", errors="ignore") as f:
    lines = f.readlines()
    print(f"Total lines in log file: {len(lines)}")


# Load station data
with gzip.open(r"C:\Users\Minamsj\FOMOsim\instances\TD_W34_old.json.gz", "rt") as f:   # adjust path if needed
    data = json.load(f)

# data["stations"] is a list of dicts
stations_df = pd.DataFrame(data["stations"])

# Split the 'location' list into separate columns
stations_df[["lat", "lon"]] = stations_df["location"].apply(pd.Series)


Total lines in log file: 401725


In [3]:
for i, line in enumerate(lines):
    stripped = line.rstrip("\n")

    # ---- New simulation ----
    if "Testing Policy" in line:
        print(f"Detected new run at line {i + 1}")

        # close previous run
        if current_run is not None:
            runs.append(current_run)

        current_run = {
            "run_id": len(runs) + 1,
            "alpha": None,
            "lost_trips": []
        }

        # extract alpha from this line
        m = alpha_pattern.search(stripped)
        if m:
            current_run["alpha"] = float(m.group(1))


    # ---- If inside a run ----
    if current_run is not None:

        # ---- CASE 1: LOST TRIP (starvation types) ----
        if "LOST TRIP" in line:
            # station Sxx
            station = None
            ms = station_pattern.search(line)
            if ms:
                station = "S" + ms.group(1)

            # type from (..., t=xx)
            lt_type = None
            mt = lost_trip_type_pattern.search(line)
            if mt:
                lt_type = mt.group(1).strip().lower()
            else:
                # fallback guess
                if "maintenance starvation" in line.lower():
                    lt_type = "maintenance starvation"
                elif "bike starvation" in line.lower():
                    lt_type = "bike starvation"

            current_run["lost_trips"].append({
                "line_no": i + 1,
                "station": station,
                "lost_trip_type": lt_type,
                "log_line": stripped
            })

        # ---- CASE 2: CONGESTION ----
        if "[CONGESTED]" in line:
            # station will be the one shown first: [S51 ARRIVAL]
            station = None
            ms = re.search(r"\[(S\d+)\s", line)
            if ms:
                station = ms.group(1)

            lt_type = "congestion"   # normalized type

            current_run["lost_trips"].append({
                "line_no": i + 1,
                "station": station,
                "lost_trip_type": lt_type,
                "log_line": stripped
            })

# close final run
if current_run is not None:
    runs.append(current_run)


# ---------- SAVE ----------
output_dir = r"C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output"
os.makedirs(output_dir, exist_ok=True)

for run in runs:
    if not run["lost_trips"]:
        continue

    df = pd.DataFrame(run["lost_trips"])
    df.insert(0, "alpha", run["alpha"])
    df.insert(0, "run_id", run["run_id"])

    alpha_str = f"{run['alpha']:.3f}" if run["alpha"] is not None else "unknown"
    out_name = os.path.join(
        output_dir,
        f"lost_trips_run_{run['run_id']}_alpha_{alpha_str}.csv"
    )
    df.to_csv(out_name, index=False)
    print("Saved:", out_name)


Detected new run at line 3
Detected new run at line 204908
Detected new run at line 392476
Saved: C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output\lost_trips_run_1_alpha_0.500.csv
Saved: C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output\lost_trips_run_2_alpha_0.400.csv
Saved: C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output\lost_trips_run_3_alpha_0.300.csv


# Logging statistics

In [23]:


output_dir = r"C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output"
csv_files = glob.glob(os.path.join(output_dir, "*.csv"))

df_combined_across_alpha = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

alpha_04 = pd.read_csv(r"C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output\lost_trips_run_2_alpha_0.400.csv")

alpha_05 = pd.read_csv(r"C:\Users\Minamsj\FOMOsim\policies\sjovik_sund\output\lost_trips_run_2_alpha_0.500.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\Minamsj\\FOMOsim\\policies\\sjovik_sund\\output\\lost_trips_run_2_alpha_0.500.csv'

In [5]:

# alpha_04: make sure there is an "id" column (numeric station id)
if "id" not in alpha_04.columns:
    if "station" in alpha_04.columns:
        alpha_04["id"] = (
            alpha_04["station"]
            .astype(str)
            .str.extract(r"S(\d+)")
            .astype(int)
        )
    else:
        raise ValueError("alpha_04 must have either 'id' or 'station' column.")

# stations_df: ensure there is an "id" column
if "id" not in stations_df.columns:
    if "Station ID" in stations_df.columns:
        stations_df["id"] = (
            stations_df["Station ID"]
            .astype(str)
            .str.extract(r"S(\d+)")
            .astype(int)
        )
    elif "station" in stations_df.columns:
        stations_df["id"] = (
            stations_df["station"]
            .astype(str)
            .str.extract(r"S(\d+)")
            .astype(int)
        )
    else:
        raise ValueError("stations_df must have 'id', 'Station ID', or 'station' column as a base for id.")

# Sanity check
print("alpha_04 columns:", alpha_04.columns.tolist())
print("stations_df columns:", stations_df.columns.tolist())

alpha_04 columns: ['run_id', 'alpha', 'line_no', 'station', 'lost_trip_type', 'log_line', 'id']
stations_df columns: ['id', 'location', 'is_depot', 'capacity', 'num_bikes', 'leave_intensities', 'leave_intensities_stdev', 'arrive_intensities', 'arrive_intensities_stdev', 'move_probabilities', 'lat', 'lon']


In [6]:
# Total lost trips per station
station_total = (
    alpha_04
    .groupby("id", as_index=False)
    .size()
    .rename(columns={"size": "n_lost_trips"})
)

# Lost trips per type per station
station_type_counts = (
    alpha_04
    .groupby(["id", "lost_trip_type"])
    .size()
    .reset_index(name="count")
    .pivot(index="id", columns="lost_trip_type", values="count")
    .fillna(0)
    .reset_index()
)

print("station_total head:\n", station_total.head())
print("station_type_counts head:\n", station_type_counts.head())

station_total head:
    id  n_lost_trips
0   1             2
1   3             3
2   4             4
3   7            31
4   8            16
station_type_counts head:
 lost_trip_type  id  bike starvation  congestion  maintenance starvation
0                1              2.0         0.0                     0.0
1                3              3.0         0.0                     0.0
2                4              0.0         0.0                     4.0
3                7              0.0         0.0                    31.0
4                8              0.0         0.0                    16.0


In [7]:
stations_lost_04 = (
    stations_df
    .merge(station_total, on="id", how="left")
    .merge(station_type_counts, on="id", how="left")
)

# Replace NaNs (stations with no lost trips) with 0
stations_lost_04["n_lost_trips"] = stations_lost_04["n_lost_trips"].fillna(0).astype(int)

for col in station_type_counts.columns:
    if col != "id":
        stations_lost_04[col] = stations_lost_04[col].fillna(0).astype(int)

print("stations_lost_04 columns:", stations_lost_04.columns.tolist())
print(stations_lost_04.head())

stations_lost_04 columns: ['id', 'location', 'is_depot', 'capacity', 'num_bikes', 'leave_intensities', 'leave_intensities_stdev', 'arrive_intensities', 'arrive_intensities_stdev', 'move_probabilities', 'lat', 'lon', 'n_lost_trips', 'bike starvation', 'congestion', 'maintenance starvation']
   id                                  location  is_depot  capacity  \
0   0    [63.40772802863199, 10.39705323440262]     False        21   
1   1  [63.414746566093925, 10.397386363673064]     False        15   
2   2  [63.436025749373755, 10.430998463879718]     False        24   
3   3   [63.42126270770024, 10.386516005020496]     False        18   
4   4   [63.43337733865644, 10.401074857528158]     False        18   

   num_bikes                                  leave_intensities  \
0         10  [[0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.2, 0.8, 0.0,...   
1          7  [[0.0, 0.0, 0.0, 0.0, 0.0, 0.2, 0.2, 0.4, 0.0,...   
2         12  [[0.0, 0.0, 0.0, 0.0, 0.2, 1.8, 0.4, 0.2, 0.0,...   
3          8  

In [20]:
def lost_color(n):
    if n >= 80:
        return "red"
    elif n >= 50:
        return "orange"
    elif n >= 10:
        return "yellow"
    elif n > 0:
        return "green"
    else:
        return "green"

def make_map_for_metric(stations_df_metric, metric_col, filename, title=None, skip_zero=True):
    """
    stations_df_metric: DataFrame with id, lat, lon, and metric_col
    metric_col: column to visualize (e.g. 'n_lost_trips' or 'congestion')
    filename: HTML file to save
    title: optional title text
    skip_zero: if True, skip stations where metric == 0
    """
    center_lat = stations_df_metric["lat"].mean()
    center_lon = stations_df_metric["lon"].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    for _, row in stations_df_metric.iterrows():
        n = int(row[metric_col]) if not pd.isna(row[metric_col]) else 0
        if skip_zero and n == 0:
            continue

        color = lost_color(n)

        popup_html = f"Station S{row['id']}<br> # Lost trips: {n}"
        if "n_lost_trips" in stations_df_metric.columns and metric_col != "n_lost_trips":
            popup_html += f"<br>Total lost trips: {int(row['n_lost_trips'])}"

        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6 + min(n, 50) / 5,  # scale with metric
            fill=True,
            fill_opacity=0.8,
            color=color,
            fill_color=color,
            popup=popup_html,
        ).add_to(m)

    m.save(filename)
    print("Saved map:", filename)
    return m

In [21]:
m_all = make_map_for_metric(
    stations_lost_04,
    metric_col="n_lost_trips",
    filename="alpha_04_lost_trips_all.html",
    title="All lost trips (alpha=0.4)"
)

# In Jupyter, show last map:
m_all

Saved map: alpha_04_lost_trips_all.html


In [22]:
# lost types are the columns from station_type_counts, excluding 'id'
lost_types = [c for c in station_type_counts.columns if c != "id"]
print("Lost trip types:", lost_types)

maps_by_type = {}

for lt in lost_types:
    safe_name = lt.replace(" ", "_")  # for nicer filenames
    fname = f"alpha_04_lost_trips_{safe_name}.html"
    m_type = make_map_for_metric(
        stations_lost_04,
        metric_col=lt,
        filename=fname,
        title=f"{lt} (alpha=0.4)"
    )
    maps_by_type[lt] = m_type

Lost trip types: ['bike starvation', 'congestion', 'maintenance starvation']
Saved map: alpha_04_lost_trips_bike_starvation.html
Saved map: alpha_04_lost_trips_congestion.html
Saved map: alpha_04_lost_trips_maintenance_starvation.html
